In [2]:
## PACKAGES
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
#import umap.umap_ as umap
import torch
import torchvision.transforms as transforms
import timm
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import seaborn as sn
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import sys
sys.path.insert(1,'/home/shelia/AllFiles/Research_tuly/pyDeepInsight/pyDeepInsight_SRT/examples/Scripts')
from pyDeepInsightv3 import ImageTransformer, CAMFeatureSelector

c:\Users\sheli\anaconda3\envs\SheliaWorkspace\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
##LOAD DATA
expr_file = r"../data/TvsN/TCGA_BRCA_NvT.csv"
expr1 = pd.read_csv(expr_file, encoding='utf-8', engine='python')
expr1=expr1.transpose()
#expr1.columns=expr1.iloc[0]
X_all = expr1.iloc[0:,0:-1]
#X_all=X_all.astype(float)
#days= expr1.iloc[1:,-1].values
#genes1 = expr1.iloc[:,:-1].columns.to_numpy()


In [42]:
X_all[0]

gene                       A1BG
TCGA-BH-A0B6-01A    0.611547418
TCGA-BH-A208-11A    0.284607255
TCGA-AR-A1AY-01A    0.206648283
TCGA-E2-A1LS-11A    0.031879073
                       ...     
TCGA-D8-A1XG-01A    0.468327486
TCGA-BH-A0H5-11A    0.455375286
TCGA-BH-A0B3-11B    0.706652806
TCGA-A8-A09V-01A    0.969028119
TCGA-BH-A1FD-01A     0.31861812
Name: 0, Length: 251, dtype: object

In [34]:
expr_file = r"../data/TvsN/TCGA_PAM50.csv"
expr2 = pd.read_csv(expr_file, encoding='utf-8', engine='python')
#expr2=expr2.transpose()
#expr2.columns=expr2.iloc[0]
X_all2 = expr2.iloc[0:,0:-1]

#days= expr2.iloc[1:,-1].values
#genes2 = expr2.iloc[:,:-1].columns.to_numpy()


In [41]:
X_all2['CLID']

0       TCGA-A1-A0SB-01A
1       TCGA-A7-A5ZV-01A
2       TCGA-AR-A24U-01A
3       TCGA-A2-A04W-01A
4       TCGA-A2-A0EM-01A
              ...       
1193    TCGA-E9-A1N5-11A
1194    TCGA-BH-A1FG-11B
1195    TCGA-E2-A1L6-01A
1196    TCGA-A7-A5ZW-01A
1197    TCGA-AC-A23E-01A
Name: CLID, Length: 1198, dtype: object

In [68]:
mask=(X_all.index.isin(X_all2['CLID']))
mask

array([False,  True,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [71]:
mask.size

251

In [70]:
# Add a new column 'PAM50_Labels' to X_new with values from X_all2_new['PAM50 and Claudin-low (CLOW) Molecular Subtype']
X_new = X_new.copy()
X_new['PAM50_Labels'] = X_all2_new.set_index('CLID').loc[X_new.index, 'PAM50 and Claudin-low (CLOW) Molecular Subtype']
X_new

,0,1,2,3,4,5,6,7,8,9,...,22479,22480,22481,22482,22483,22484,22485,22486,22487,22488
TCGA-BH-A0B6-01A,0.611547418,4.548901716,0.005932882,298.7804831,2.261264314,0.819144698,0.097871529,18.73948802,26.93090763,7.678354891,...,16.41414641,7.719440278,21.05957735,6.690538088,13.2525652,12.7313273,0.849625583,8.965085225,226.1921791,9.275500987
TCGA-BH-A208-11A,0.284607255,3.815896617,0.082592789,562.2490719,4.251541036,0.447993498,0.255466572,9.988344367,31.93733947,14.16230835,...,30.76659186,8.669074675,8.254911307,6.725190104,15.92097126,19.36489219,2.333878613,25.72887081,269.6944406,24.33601126
TCGA-AR-A1AY-01A,0.206648283,3.606146947,0,2901.954132,1.573587707,43.47171896,0.103693922,5.602335826,41.94514018,14.03811536,...,86.83208436,22.53535961,69.13113572,6.239441685,14.77102806,31.89094405,20.10502011,35.29246656,279.2133536,12.03932748
TCGA-A2-A0YE-01A,0.675570554,3.690335864,0.017988631,147.8324838,3.677657529,1.616353541,0.0865517,5.028363437,26.0494859,20.9267325,...,29.94551608,30.93423227,70.87526092,4.045972549,11.77228355,9.572150088,1.349377295,19.17513948,135.817851,9.864087249
TCGA-BH-A0BS-11A,0.215181655,4.876433882,0.017953092,648.5477026,3.801477132,1.211837396,0.357862911,14.89907155,34.566813,15.64708011,...,38.06860747,8.675286512,13.84738291,6.077926968,13.2603109,21.41862683,1.750724793,23.67972536,306.0385682,15.82374536
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-D8-A1XG-01A,0.468327486,5.270435175,0.015028316,518.6841294,1.975140275,0.197613195,0.015494623,23.90570731,34.57956074,12.97543045,...,36.68215732,14.86621976,40.23638158,9.929902082,29.30901741,21.22480675,0.21521483,16.90856281,172.8321483,27.52183033
TCGA-BH-A0H5-11A,0.455375286,5.200037361,0.080369786,804.6963613,3.136841537,0.557229047,0.451982969,13.99441966,36.9465421,14.28193165,...,35.36281317,9.65118641,11.77248375,9.177631593,17.86806875,22.13509747,4.319781813,28.63683893,223.3845649,26.55741062
TCGA-BH-A0B3-11B,0.706652806,5.973488174,0.037632537,790.3604149,4.836055684,1.047421768,0.944138694,22.61471874,35.93765858,9.895018914,...,30.39726663,11.20401104,12.33491854,8.434958472,16.47305243,20.90929124,2.797258377,30.06400853,195.8454827,28.35152804
TCGA-A8-A09V-01A,0.969028119,7.282172794,0,201.4171671,1.671179996,0.14973305,0.105663562,21.88487165,47.92322595,11.11573728,...,34.8662915,21.49115161,51.80102068,11.17959992,32.69926309,28.65663848,10.31999798,39.71575542,99.69319989,30.53991563


In [76]:
X_all2_new = X_all2[X_all2['CLID'].isin(X_new.index)]
X_all2_new

,CLID,2016 Histology Annotations,PAM50 and Claudin-low (CLOW) Molecular Subtype,Tumor or Normal,er_status_by_ihc,pr_status_by_ihc,HER2.newly.derived,Triple Negative Status,12 Biological group annotations,Special Comments: original histologic type diagnosis (as provided from Pathology forms at time of inclusion into TCGA),Special Comments: original histologic diagnosis for special histologic and mixed histologic type (as provided from Pathology forms at time of inclusion into TCGA)
2,TCGA-AR-A24U-01A,Carcinoma with apocrine differentiation,HER2E,Tumor,Negative,Negative,Positive,No,-,Infiltrating Ductal Carcinoma,[Not Applicable]
5,TCGA-C8-A26Z-01A,Cribriform carcinoma,LumA,Tumor,Positive,Positive,Negative,No,Cribriform carcinoma,Infiltrating Ductal Carcinoma,[Not Applicable]
10,TCGA-E9-A5FL-01A,Invasive ductal carcinoma,Basal,Tumor,Negative,Negative,Negative,Yes,Invasive ductal carcinoma - Basal,Metaplastic Carcinoma,[Not Applicable]
33,TCGA-AN-A0AR-01A,Invasive ductal carcinoma,Basal,Tumor,Negative,Negative,Negative,Yes,Invasive ductal carcinoma - Basal,Infiltrating Ductal Carcinoma,[Not Applicable]
38,TCGA-AR-A1AY-01A,Invasive ductal carcinoma,Basal,Tumor,Negative,Negative,Negative,Yes,Invasive ductal carcinoma - Basal,Infiltrating Ductal Carcinoma,[Not Applicable]
...,...,...,...,...,...,...,...,...,...,...,...
1190,TCGA-BH-A0DZ-11A,True Normal,True Normal,normal,Not Applicable,Not Applicable,Not Applicable,Not Applicable,-,na,na
1191,TCGA-BH-A0E0-11A,True Normal,True Normal,normal,Not Applicable,Not Applicable,Not Applicable,Not Applicable,-,na,na
1192,TCGA-BH-A0DK-11A,True Normal,True Normal,normal,Not Applicable,Not Applicable,Not Applicable,Not Applicable,-,na,na
1193,TCGA-E9-A1N5-11A,True Normal,True Normal,normal,Not Applicable,Not Applicable,Not Applicable,Not Applicable,-,na,na


In [ ]:
# Add a new column 'PAM50_Labels' to X_new with values from X_all2_new['PAM50 and Claudin-low (CLOW) Molecular Subtype']
X_new2 = X_new.copy()
X_new2['PAM50_Labels'] = X_all2_new.set_index('CLID').loc[X_new2.index, 'PAM50 and Claudin-low (CLOW) Molecular Subtype']
X_new2

In [80]:
# Save X_new2 as a tab-delimited txt file


In [83]:
X_new3=X_new2.transpose()
X_new3.to_csv('X_new2_export.csv')


In [84]:
X_new2.to_csv('X_new2_export.txt', sep='\t')